In [1]:
import numpy as np
import pandas as pd
import Soil_Carbon_Model as md
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit 
import os
import openpyxl
from pandas import *
%matplotlib inline

In [2]:
import pylab
params = {'backend': 'ps',
          'axes.labelsize': 20,
          'legend.fontsize': 20,
          'legend.handlelength': 2.0,
          'legend.borderaxespad': 0,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'font.family': 'serif',
          'font.size': 20,
          'font.weight': 'normal',
          #'font.serif': ['Palatino'],
          'ps.usedistiller': 'xpdf',
          'text.usetex': False,
         'agg.path.chunksize' : 1000}
pylab.rcParams.update(params)

In [3]:
dt = 1 / 24
nyears = 2000
ndays = nyears * 365
nsim = ndays * 24
nnpp = ndays * 24

#Rainfall from Poisson
rain = md.poisson_process(n = int(nsim), alpha = 9.15/1000, lbd = (0.147 * dt))

tsim = np.linspace(1, nnpp, int(nnpp)) # Hour
"""

npp_original = np.array([0.82])
npp = np.tile(npp_original,nsim)

"""
days = np.arange(365)
annual_avg_npp = 306.6

# Half sine wave shape over 365 days
shape = np.sin((np.pi*days)/365)                       # sinB(x-h) + K; B = pi/365, h = 0, K = 0

# Normalize so that the sum equals the annual NPP
daily_npp = annual_avg_npp * (shape / np.sum(shape))
hourly_npp = np.repeat(daily_npp, 24)

npp = np.tile(hourly_npp,nyears)

In [4]:
print("Maximum Value is:", max(npp))
print("Length is:", len(npp))
print("Average value is:", np.mean(npp))

Maximum Value is: 1.319464841628453
Length is: 17520000
Average value is: 0.8400000000000017


In [5]:
total_rainfall = rain.sum()
print(f"Total rainfall over 450 years = {total_rainfall:.3f} meters")

RPY = (total_rainfall / nyears)*1000         # average rainfall per year
print(f"Average rainfall per year = {RPY:.3f} mm/year")

Total rainfall over 450 years = 985.042 meters
Average rainfall per year = 492.521 mm/year


In [6]:
PET = 972.6
AI = PET/RPY
print(AI)

"""
def hill_constrained(R, KA, n):
    R_max = 4.18

    numerator = R**n
    denominator = KA**n + R**n
    norm_factor = R_max**n / (KA**n + R_max**n)
    TI = (numerator / denominator) / norm_factor

    # Clip to ensure TI does not exceed 1
    return (np.minimum(TI, 1.0))

TI = hill_constrained(AI, 3.45, 2.28)
print(TI)
"""

# Final best-fit parameters from your latest run
KA_final = 9.0132
n_final = 1.7444
R_max = 3.8  # Max AI used for normalization

def compute_TI(AI):
    AI = np.asarray(AI)  # Accept scalars or arrays
    numerator = AI**n_final
    denominator = KA_final**n_final + AI**n_final
    norm_factor = R_max**n_final / (KA_final**n_final + R_max**n_final)
    TI = (numerator / denominator) / norm_factor
    return np.minimum(TI, 1.0)

TI = compute_TI(AI)
print("TI =", TI)

1.9747379124310163
TI = 0.36422454596919795


In [7]:
#Hydraulic parameterization for dynamic porosity -------------------------------
alpha1 = 580  #hydraulic conductivity shape parameter
beta1 = 6     #hydraulic conductivity shape parameter
alpha2 = 580  #porosity shape parameter
beta2 = 6     #porosity shape parameter
kstex = 0.1   #m/d,
kstot = 1     #m/d
phitex = 0.50  
phitot = 0.60  

In [8]:
# Simulation for 2150 years at dynamic porosity
sim = md.run_julia(rain, npp, kstex, kstot, phitex, phitot, alpha1, beta1, alpha2, beta2, TI)
sim_year = sim.pivot_table(values=['s', 'phi', 'ksstr', 'P', 'DOC', 'A', 'M', 'B',
                                   'Fa', 'Fpa','Flb','Fl', 'Flm', 'Fpl', 'Fbm','Fld', 'Fal',
                                   'lk','npp_', 'L','Fi', 'Flpom', 'decomp'],
                           index='Years')

print(sim_year)

100%|██████████| 17520000/17520000 [21:08<00:00, 13808.85it/s]


                 A           B         DOC        Fa       Fal       Fbm  \
Years                                                                      
1       873.211748   87.541761   39.488014  0.050966  0.001481  0.211055   
2       748.973631   40.979331   35.333051  0.046070  0.001072  0.098797   
3       736.875400   19.140136   32.014038  0.049807  0.000809  0.046145   
4       745.053572    8.896837   30.554544  0.048920  0.000489  0.021449   
5       712.475170    4.109256   36.087513  0.041841  0.000219  0.009907   
...            ...         ...         ...       ...       ...       ...   
1996   1807.467840  104.263572  239.899070  0.123206  0.003722  0.251348   
1997   1807.702196  105.063458  242.134659  0.122908  0.003719  0.253277   
1998   1807.836860  100.679201  265.775908  0.111176  0.003337  0.242707   
1999   1807.979487  103.730126  242.034442  0.129586  0.003912  0.250062   
2000   1808.067634  102.439016  253.553729  0.102202  0.003080  0.246952   

           

In [9]:
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Lou\LOUChangedSoilTexture.csv", index= True, header = True)

sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Results\LOUChangedSoilTexture.csv", index= True, header = True)